In [1]:
# imports

import kagglehub
import os
import pandas as pd

In [2]:
# Load the dataset

path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")
print("Path:", path)
print("Files:", os.listdir(path))

Path: /Users/gracescanlan/.cache/kagglehub/datasets/alexteboul/diabetes-health-indicators-dataset/versions/1
Files: ['diabetes_012_health_indicators_BRFSS2015.csv', 'diabetes_binary_health_indicators_BRFSS2015.csv', 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv']


In [3]:
# Read the CSV file into a DataFrame and check its shape
data = pd.read_csv(os.path.join(path, "diabetes_binary_health_indicators_BRFSS2015.csv"))
data.shape

(253680, 22)

In [4]:
# Display the first few rows of the dataset
data.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


# Key to data
 - Diabetes: 0 = no diabetes 1 = prediabetes 2 = diabetes
 - HighBP: 0 = no high BP 1 = high BP
 - HighChol: 0 = no high cholesterol 1 = high cholesterol
 - CholCheck: 0 = no cholesterol check in 5 years 1 = yes cholesterol check in 5 years
 - Smoker: Have you smoked at least 100 cigarettes in your entire life? [Note: 5 packs = 100 cigarettes] 0 = no 1 = yes
 - Stroke: (Ever told) you had a stroke. 0 = no 1 = yes
 - HeartDisease: coronary heart disease (CHD) or myocardial infarction (MI) 0 = no 1 = yes
 - PhysActivity: physical activity in past 30 days - not including job 0 = no 1 = yes
 - Fruits: Consume Fruit 1 or more times per day 0 = no 1 = yes




In [5]:
# cleaning data

# standardize column names
data.columns = data.columns.str.strip().str.lower().str.replace(" ", "_")

In [6]:
# check for duplicates
duplicates = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

Number of duplicate rows: 24206


In [7]:
# drop duplicates if any
data = data.drop_duplicates()

In [8]:
# Handle missing values if any
missing_values = data.isnull().sum()
print("Missing values in each column:\n", missing_values)

Missing values in each column:
 diabetes_binary         0
highbp                  0
highchol                0
cholcheck               0
bmi                     0
smoker                  0
stroke                  0
heartdiseaseorattack    0
physactivity            0
fruits                  0
veggies                 0
hvyalcoholconsump       0
anyhealthcare           0
nodocbccost             0
genhlth                 0
menthlth                0
physhlth                0
diffwalk                0
sex                     0
age                     0
education               0
income                  0
dtype: int64


In [ ]:
# data types of each column
data.dtypes

diabetes_binary         float64
highbp                  float64
highchol                float64
cholcheck               float64
bmi                     float64
smoker                  float64
stroke                  float64
heartdiseaseorattack    float64
physactivity            float64
fruits                  float64
veggies                 float64
hvyalcoholconsump       float64
anyhealthcare           float64
nodocbccost             float64
genhlth                 float64
menthlth                float64
physhlth                float64
diffwalk                float64
sex                     float64
age                     float64
education               float64
income                  float64
dtype: object

In [12]:
binary_cols = [
    'diabetes_binary', 'highbp', 'highchol', 'cholcheck',
    'smoker', 'stroke', 'heartdiseaseorattack',
    'physactivity', 'fruits', 'veggies',
    'hvyalcoholconsump', 'anyhealthcare',
    'nodocbccost', 
    'diffwalk', 'sex'
]

data[binary_cols] = data[binary_cols].astype(int)

In [13]:
# Remove impossible values

data = data[(data['age'] >= 0) & (data['age'] <= 100)]
data = data[(data['bmi'] >= 10) & (data['bmi'] <= 60)]

In [15]:
# Normalize / scale data

from sklearn.preprocessing import StandardScaler

features = data.drop(columns=['diabetes_binary'])
scaler = StandardScaler()
data_scaled = scaler.fit_transform(features)

In [16]:
# check class balance

class_counts = data['diabetes_binary'].value_counts()
print("Class distribution:\n", class_counts)

Class distribution:
 diabetes_binary
0    193743
1     34926
Name: count, dtype: int64


In [18]:
# save cleaned data to a new CSV file 

data.to_csv("../data/cleaned_diabetes_data.csv", index=False)